In [21]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "anndata>=0.10.0",
#     "anywidget>=0.9.0",
#     "ipywidgets>=8.0.0",
#     "jupyter-scatter>=0.21.0,<1.0.0",
#     "matplotlib>=3.7.0",
#     "numpy>=1.24.0",
#     "pandas>=2.0.0",
#     "scanpy>=1.9.0",
#     "scipy>=1.11.0",
#     "traitlets>=5.0.0",
# ]
#
# [tool.uv.sources]
# scsketch = { path = "../..", editable = true }
# ///


# scSketch UMAP-PCA linked-view supplementary figure

Purpose: create a manuscript-revision figure showing that a scSketch selection made on a UMAP view can be inspected in an independent PCA projection of the same cells.

Use a small, visually simple trajectory dataset or a filtered subset of a larger dataset. The live linked views are for selecting and inspecting cells; the static Matplotlib figure generated later is the figure panel intended for the supplement.


## Interpretation note

This linked view is an embedding diagnostic. It does not prove that UMAP distances are quantitatively faithful and does not turn scSketch into a trajectory-inference method. It helps document whether the same user-selected cells form a coherent structure in a second projection.


In [22]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display
from jscatter import Scatter

from scsketch import ScSketch


## Configure inputs

Set `DATA_FILENAME` to the `.h5ad` file you want to use. Put candidate datasets in `Manuscript/revision_analyses/data/` when possible.

For a simpler figure, either choose a simple dataset or set `SUBSET_COL` / `SUBSET_VALUES` to keep one clean lineage, branch, time course, or cell-state transition.


In [23]:
DATA_FILENAME = "trajectory_task_dataset_v1_participant.h5ad"
DATASET_NAME = Path(DATA_FILENAME).stem

# Optional: restrict a complex dataset to one visually simpler trajectory.
# Example: SUBSET_COL = "lineage"; SUBSET_VALUES = ["AB", "MS"]
SUBSET_COL = None
SUBSET_VALUES = None

# Optional: override after inspecting adata.obs columns.
LABEL_COL = None

N_TOP_GENES = 2000
N_PCS = 50
N_NEIGHBORS = 30
UMAP_RANDOM_STATE = 0

DATA_PATH_CANDIDATES = [
    Path("data") / DATA_FILENAME,
    Path("Manuscript/revision_analyses/data") / DATA_FILENAME,
]
DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    checked = "; ".join(str(path) for path in DATA_PATH_CANDIDATES)
    raise FileNotFoundError(f"Could not find {DATA_FILENAME}. Checked: {checked}")

REVISION_DIR = DATA_PATH.parent.parent
OUTPUT_DIR = REVISION_DIR / "outputs" / DATASET_NAME / "umap_pca_linked_view"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using AnnData: {DATA_PATH}")
print(f"Writing outputs to: {OUTPUT_DIR}")


Using AnnData: data/trajectory_task_dataset_v1_participant.h5ad
Writing outputs to: outputs/trajectory_task_dataset_v1_participant/umap_pca_linked_view


## Load AnnData and optionally subset


In [24]:
adata_raw = sc.read_h5ad(DATA_PATH)
print(adata_raw)
print("obs columns:", list(adata_raw.obs.columns))
print("obsm keys:", list(adata_raw.obsm.keys()))


AnnData object with n_obs × n_vars = 6188 × 20222
    obs: 'cell', 'n.umi', 'time.point', 'batch', 'Size_Factor', 'raw.embryo.time', 'embryo.time', 'embryo.time.bin', 'raw.embryo.time.bin', 'lineage', 'num_genes_expressed', 'cell.type', 'bg.300.loading', 'bg.400.loading', 'bg.500.1.loading', 'bg.500.2.loading', 'bg.r17.loading', 'bg.b01.loading', 'bg.b02.loading'
    var: 'id', 'gene_short_name', 'num_cells_expressed'
    uns: 'download_files', 'source', 'source_url'
    layers: 'counts'
obs columns: ['cell', 'n.umi', 'time.point', 'batch', 'Size_Factor', 'raw.embryo.time', 'embryo.time', 'embryo.time.bin', 'raw.embryo.time.bin', 'lineage', 'num_genes_expressed', 'cell.type', 'bg.300.loading', 'bg.400.loading', 'bg.500.1.loading', 'bg.500.2.loading', 'bg.r17.loading', 'bg.b01.loading', 'bg.b02.loading']
obsm keys: []


In [25]:
adata = adata_raw.copy()

if SUBSET_COL is not None and SUBSET_VALUES is not None:
    if SUBSET_COL not in adata.obs.columns:
        raise KeyError(f"SUBSET_COL={SUBSET_COL!r} is not in adata.obs.")
    keep = adata.obs[SUBSET_COL].astype(str).isin([str(value) for value in SUBSET_VALUES])
    adata = adata[keep].copy()
    print(f"Subset {SUBSET_COL} to {SUBSET_VALUES}: {adata.n_obs:,} cells")
else:
    print(f"Using all cells: {adata.n_obs:,}")

if adata.n_obs < 50:
    raise ValueError("The selected subset is very small. Pick a broader subset for visualization.")

adata


Using all cells: 6,188


AnnData object with n_obs × n_vars = 6188 × 20222
    obs: 'cell', 'n.umi', 'time.point', 'batch', 'Size_Factor', 'raw.embryo.time', 'embryo.time', 'embryo.time.bin', 'raw.embryo.time.bin', 'lineage', 'num_genes_expressed', 'cell.type', 'bg.300.loading', 'bg.400.loading', 'bg.500.1.loading', 'bg.500.2.loading', 'bg.r17.loading', 'bg.b01.loading', 'bg.b02.loading'
    var: 'id', 'gene_short_name', 'num_cells_expressed'
    uns: 'download_files', 'source', 'source_url'
    layers: 'counts'

## Pick metadata for coloring

The notebook tries common single-cell annotation columns first. Set `LABEL_COL` in the configuration cell if the automatic choice is not the column you want in the figure.


In [26]:
CANDIDATE_METADATA_COLS = [
    "cell_type",
    "celltype",
    "cell_type_short",
    "cell_state",
    "cell.type",
    "lineage",
    "timepoint",
    "time.point",
    "embryo.time.bin",
    "embryo.time",
    "cluster",
    "clusters",
    "louvain",
    "leiden",
    "seurat_clusters",
    "partition",
    "time",
]

metadata_cols = [col for col in CANDIDATE_METADATA_COLS if col in adata.obs.columns]

if LABEL_COL is None:
    COLOR_BY = metadata_cols[0] if metadata_cols else None
else:
    if LABEL_COL not in adata.obs.columns:
        raise KeyError(f"LABEL_COL={LABEL_COL!r} is not in adata.obs.")
    COLOR_BY = LABEL_COL
    if COLOR_BY not in metadata_cols:
        metadata_cols = [COLOR_BY] + metadata_cols

print("Available metadata columns selected for scSketch:", metadata_cols)
print("Coloring by:", COLOR_BY)
print("First obs columns:", list(adata.obs.columns[:20]))


Available metadata columns selected for scSketch: ['cell.type', 'lineage', 'time.point', 'embryo.time.bin', 'embryo.time']
Coloring by: cell.type
First obs columns: ['cell', 'n.umi', 'time.point', 'batch', 'Size_Factor', 'raw.embryo.time', 'embryo.time', 'embryo.time.bin', 'raw.embryo.time.bin', 'lineage', 'num_genes_expressed', 'cell.type', 'bg.300.loading', 'bg.400.loading', 'bg.500.1.loading', 'bg.500.2.loading', 'bg.r17.loading', 'bg.b01.loading', 'bg.b02.loading']


## Load Monocle UMAP and compute PCA

The UMAP for this reviewer figure should come from the Monocle 3 tutorial workflow, not from a new Scanpy UMAP run. First generate the coordinate CSV from the repository root:

```bash
Rscript Manuscript/revision_analyses/export_monocle_tutorial_umap.R
```

This notebook loads those Monocle coordinates into `adata.obsm["X_umap"]`. PCA is still computed in Python only for the linked comparison view.


In [ ]:
UMAP_COORDS_CANDIDATES = [
    Path("outputs/monocle_umap/monocle_tutorial_umap.csv"),
    Path("Manuscript/revision_analyses/outputs/monocle_umap/monocle_tutorial_umap.csv"),
]
UMAP_COORDS_PATH = next((path for path in UMAP_COORDS_CANDIDATES if path.exists()), None)

if UMAP_COORDS_PATH is None:
    checked = "\n".join(f"- {path}" for path in UMAP_COORDS_CANDIDATES)
    raise FileNotFoundError(
        "Monocle UMAP coordinates were not found. Run this from the repo root first:\n"
        "Rscript Manuscript/revision_analyses/export_monocle_tutorial_umap.R\n\n"
        f"Checked:\n{checked}"
    )

umap_df = pd.read_csv(UMAP_COORDS_PATH)
required_cols = {"cell", "UMAP1", "UMAP2"}
missing_cols = required_cols.difference(umap_df.columns)
if missing_cols:
    raise ValueError(f"Missing columns in {UMAP_COORDS_PATH}: {sorted(missing_cols)}")
if umap_df["cell"].duplicated().any():
    raise ValueError(f"Duplicate cell IDs found in {UMAP_COORDS_PATH}")

umap_df = umap_df.set_index("cell")
missing_cells = adata.obs_names[~adata.obs_names.isin(umap_df.index)]
if len(missing_cells) > 0:
    preview = ", ".join(map(str, missing_cells[:5]))
    raise ValueError(
        f"{len(missing_cells):,} AnnData cells are missing from the Monocle UMAP CSV. "
        f"First missing cells: {preview}"
    )

adata.obsm["X_umap"] = umap_df.loc[adata.obs_names, ["UMAP1", "UMAP2"]].to_numpy(dtype=float)
adata.obsm["X_umap_monocle"] = adata.obsm["X_umap"].copy()
adata.uns["X_umap_source"] = "Monocle 3 tutorial workflow exported by export_monocle_tutorial_umap.R"

if "X_pca" not in adata.obsm or adata.obsm["X_pca"].shape[1] < 2:
    adata_pca = adata.copy()
    if "counts" in adata_pca.layers:
        adata_pca.X = adata_pca.layers["counts"].copy()

    sc.pp.normalize_total(adata_pca, target_sum=1e4)
    sc.pp.log1p(adata_pca)

    if adata_pca.n_vars > N_TOP_GENES:
        sc.pp.highly_variable_genes(adata_pca, n_top_genes=N_TOP_GENES)
        adata_pca = adata_pca[:, adata_pca.var["highly_variable"]].copy()
        print(f"Restricted PCA computation to {adata_pca.n_vars:,} highly variable genes")

    n_comps = min(N_PCS, adata_pca.n_obs - 1, adata_pca.n_vars - 1)
    if n_comps < 2:
        raise ValueError("Need at least two PCA components for linked PCA visualization.")

    sc.pp.pca(adata_pca, n_comps=n_comps)
    adata.obsm["X_pca"] = adata_pca.obsm["X_pca"]
else:
    print("Using existing adata.obsm['X_pca']")

print(f"Loaded Monocle UMAP coordinates: {UMAP_COORDS_PATH}")
print(f"X_umap source: {adata.uns['X_umap_source']}")
print(f"X_umap shape: {adata.obsm['X_umap'].shape}")
print(f"X_pca shape: {adata.obsm['X_pca'].shape}")


## Build scSketch and PCA views

The first widget is the full scSketch UI on UMAP. The second widget is a separate Jupyter Scatter PCA view. The two widgets synchronize selected cells and hovered cells, but not pan/zoom/camera state.


In [30]:
if COLOR_BY is None:
    adata.obs["all_cells"] = "all_cells"
    metadata_cols = ["all_cells"]
    COLOR_BY = "all_cells"

pca_df = pd.DataFrame(
    {
        "obs_name": adata.obs_names.astype(str),
        "PC1": np.asarray(adata.obsm["X_pca"][:, 0], dtype=float),
        "PC2": np.asarray(adata.obsm["X_pca"][:, 1], dtype=float),
        COLOR_BY: adata.obs[COLOR_BY].astype(str).to_numpy(),
    },
    index=adata.obs_names,
)

sketch = ScSketch(
    adata=adata,
    metadata_cols=metadata_cols,
    color_by_default=COLOR_BY,
    height=620,
    max_genes=0, 
)

shared_color_map = sketch.categorical_color_maps.get(COLOR_BY)
pca_color_kwargs = {}
if shared_color_map is not None:
    pca_color_kwargs["color_map"] = shared_color_map


pca = Scatter(
    data=pca_df,
    x="PC1",
    y="PC2",
    color_by=COLOR_BY,
    **pca_color_kwargs,
    height=620,
    axes=True,
    legend=False,
    tooltip=True,
    tooltip_properties=["obs_name", COLOR_BY],
)


In [31]:
syncing = {"active": False}


def _clean_selection(value):
    if value is None:
        return []
    return np.asarray(value, dtype=int).tolist()


def sync_sketch_to_pca(change):
    if syncing["active"]:
        return
    syncing["active"] = True
    try:
        pca.selection(_clean_selection(change["new"]))
    finally:
        syncing["active"] = False


def sync_pca_to_sketch(change):
    if syncing["active"]:
        return
    syncing["active"] = True
    try:
        sketch.scatter.selection(_clean_selection(change["new"]))
    finally:
        syncing["active"] = False


sketch.scatter.widget.observe(sync_sketch_to_pca, names="selection")
pca.widget.observe(sync_pca_to_sketch, names="selection")


## Interactive linked view

Use scSketch above the PCA view to draw the candidate trajectory selection. The same cells should highlight in the PCA view. Save the selection in scSketch before exporting the session below.


In [32]:
display(
    widgets.VBox(
        [sketch.show(), pca.show()],
        layout=widgets.Layout(width="100%", gap="16px"),
    )
)


## Export reproducibility artifacts

Run this after drawing and saving the scSketch selection. The session JSON captures the selected cells and drawn path. The selected-cell table is useful for downstream checks and figure captions.

In [ ]:
SESSION_PATH = OUTPUT_DIR / "umap_pca_linked_view.scsketch.json"
SELECTED_CELLS_PATH = OUTPUT_DIR / "umap_pca_linked_view_selected_cells.csv"

session = sketch.export_session(SESSION_PATH)

if sketch.active_selection is None:
    raise RuntimeError("No active scSketch selection. Draw and save a selection first.")

selected_idx = np.asarray(sketch.active_selection.points, dtype=int)
selected_cells = adata.obs.iloc[selected_idx].copy()
selected_cells.insert(0, "selection", sketch.active_selection.name)
selected_cells.insert(1, "obs_name", adata.obs_names[selected_idx].astype(str))
selected_cells["UMAP1"] = adata.obsm["X_umap"][selected_idx, 0]
selected_cells["UMAP2"] = adata.obsm["X_umap"][selected_idx, 1]
selected_cells["PC1"] = adata.obsm["X_pca"][selected_idx, 0]
selected_cells["PC2"] = adata.obsm["X_pca"][selected_idx, 1]
selected_cells.to_csv(SELECTED_CELLS_PATH, index=False)

print(f"Wrote session: {SESSION_PATH}")
print(f"Wrote selected cells: {SELECTED_CELLS_PATH}")
print(f"Active selection: {sketch.active_selection.name}")
print(f"Selected cells: {len(selected_idx):,}")


## Selected-cell summary

Use this table in the supplement or response-to-reviewers text if helpful.

In [ ]:
summary = (
    selected_cells[COLOR_BY]
    .value_counts(dropna=False)
    .rename_axis(COLOR_BY)
    .reset_index(name="n_selected_cells")
)
summary["fraction_selected"] = summary["n_selected_cells"] / len(selected_cells)
summary


## Static figure for supplement

Run this after exporting a saved scSketch selection. It makes a clean side-by-side figure with all cells in light gray and the scSketch-selected cells highlighted in blue.


In [ ]:
FIGURE_PATH = OUTPUT_DIR / "suppfig_umap_pca_linked_selection.png"
PDF_PATH = OUTPUT_DIR / "suppfig_umap_pca_linked_selection.pdf"

umap_xy = np.asarray(adata.obsm["X_umap"], dtype=float)
pca_xy = np.asarray(adata.obsm["X_pca"][:, :2], dtype=float)
selected_mask = np.zeros(adata.n_obs, dtype=bool)
selected_mask[selected_idx] = True

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.8), dpi=150)

panels = [
    (axes[0], umap_xy, "UMAP scSketch selection", "UMAP1", "UMAP2"),
    (axes[1], pca_xy, "Same cells in PCA", "PC1", "PC2"),
]

for ax, xy, title, xlabel, ylabel in panels:
    ax.scatter(
        xy[~selected_mask, 0],
        xy[~selected_mask, 1],
        s=2,
        c="#d0d0d0",
        alpha=0.45,
        linewidths=0,
    )
    ax.scatter(
        xy[selected_mask, 0],
        xy[selected_mask, 1],
        s=7,
        c="#0072B2",
        alpha=0.9,
        linewidths=0,
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

fig.suptitle(f"{DATASET_NAME}: {sketch.active_selection.name}", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURE_PATH, bbox_inches="tight", dpi=300)
fig.savefig(PDF_PATH, bbox_inches="tight")
plt.show()

print(f"Wrote figure: {FIGURE_PATH}")
print(f"Wrote PDF: {PDF_PATH}")


## Figure caption draft

Supplementary Figure 2. Linked embedding diagnostic for scSketch selections. A user-defined trajectory selection drawn in the scSketch UMAP view highlights the same cells in a PCA projection of the same AnnData object. The PCA view provides an independent projection for checking whether the selected structure is visible beyond the nonlinear UMAP layout. This diagnostic does not imply that distances in either two-dimensional projection are quantitatively faithful; instead, it illustrates how scSketch can be composed with linked notebook views to support cautious exploratory interpretation.
